<!-- open-in-badges -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/06_foundation_models/notebooks/sesion_06_3_preentrenamiento_gpt.ipynb) [![Abrir en Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/EAFIT-IA/si7011-DeepLearning/blob/main/sessions/06_foundation_models/notebooks/sesion_06_3_preentrenamiento_gpt.ipynb) [![Abrir en Lightning Studio](https://pl-bolts-doc-images.s3.us-east-2.amazonaws.com/app-2/studio-badge.svg)](https://lightning.ai/new?repo_url=https%3A%2F%2Fgithub.com%2FEAFIT-IA%2Fsi7011-DeepLearning%2Fblob%2Fmain%2Fsessions%2F06_foundation_models%2Fnotebooks%2Fsesion_06_3_preentrenamiento_gpt.ipynb)

# SI7011 · Sesión 06 · 3. Preentrenar un GPT pequeño
**Juan David Martínez Vargas · Universidad EAFIT**

Acompaña la **Parte 3** de las diapositivas. Entrenamos un modelo de lenguaje de unos 2.5 millones de parámetros sobre *Don Quijote de la Mancha*, con un tokenizador BPE entrenado aquí mismo. La única supervisión es el texto: predecir el token siguiente.

| Sección | Lo que cambia | Qué mirar |
|---|---|---|
| 1 | el texto | de dónde sale, cuánto es |
| 2 | BPE a mano, 60 fusiones | cómo se acorta una frase |
| 3 | BPE con `tokenizers`, $\lvert V\rvert$ = 300 … 16 000 | caracteres por token |
| 4 | el GPT: el bloque del 6.2 con máscara causal | pérdida inicial $\approx \ln\lvert V\rvert$ |
| 5 | entrenar unos minutos | la curva y las muestras a los 200 y 2 000 pasos |
| 6 | el *checkpoint* largo | qué llega con más entrenamiento, y qué no |
| 7 | greedy, temperatura, top-$k$ | cómo cambia el texto |

**Datos:** *Don Quijote*, Project Gutenberg (dominio público), unos 2 millones de caracteres. La celda de la sección 1 lo descarga; en Kaggle activa **Internet** en *Settings*, o pon `pg2000.txt` en `data/`. **Usa GPU** (Colab o Kaggle): la sección 5 tarda unos minutos. **Checkpoint largo:** en Kaggle agrega el modelo `juanmartinezv4399/si7011-s06-gpt-quijote` con **Add Input → Models**; en Colab la celda lo baja con tu secreto `KAGGLE_API_TOKEN`. Sin él, el notebook sigue con el modelo de la sección 5.

In [ ]:
import json, math, os, re, time, urllib.request
from pathlib import Path
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F

try:
    from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers
except ImportError:                                        # Kaggle y Colab ya la traen
    os.system('pip install -q tokenizers')
    from tokenizers import Tokenizer, models, pre_tokenizers, decoders, trainers

torch.manual_seed(0)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('PyTorch', torch.__version__, '·', dev)

## 1. El texto
Los últimos capítulos (10 %) quedan aparte para validar: el modelo nunca los ve al entrenar. Se cortan por posición y no al azar, como en las series de tiempo de la sesión 05: si se mezclaran frases de todo el libro, la validación estaría llena de frases casi repetidas del entrenamiento.

In [ ]:
GUTENBERG = 'https://www.gutenberg.org/cache/epub/2000/pg2000.txt'    # Don Quijote, dominio público
ROOTS = [Path('.'), Path('data'), Path('/kaggle/input'), Path('/teamspace/studios/this_studio/data')]

def buscar(*patrones):
    for d in ROOTS:
        for patron in patrones:
            hits = sorted(d.glob(f'**/{patron}')) if d.exists() else []
            if hits:
                return hits[0]
    return None

def cargar_texto():
    ruta = buscar('pg2000.txt', 'quijote*.txt')
    if ruta is not None:
        return ruta.read_text(encoding='utf-8'), f'archivo local ({ruta})'
    Path('data').mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(GUTENBERG, 'data/pg2000.txt')
        return Path('data/pg2000.txt').read_text(encoding='utf-8'), 'descargado de Project Gutenberg'
    except Exception as e:
        raise RuntimeError('Sin internet: pon pg2000.txt (Don Quijote, Project Gutenberg) en data/. '
                           'En Kaggle, activa Internet en Settings.') from e

crudo, fuente = cargar_texto()
inicio, fin = crudo.find('*** START OF'), crudo.find('*** END OF')
texto = crudo[crudo.find('\n', inicio) + 1:fin] if inicio >= 0 and fin > 0 else crudo   # sin la licencia de Gutenberg
texto = re.sub(r'\r\n?', '\n', texto).strip()
corte = int(0.9 * len(texto))
texto_train, texto_val = texto[:corte], texto[corte:]                # los últimos capítulos quedan para validar
print(f'fuente: {fuente} · {len(texto):,} caracteres · {len(set(texto))} caracteres distintos')
FRASE = 'En un lugar de la Mancha, de cuyo nombre no quiero acordarme'          # la primera frase del libro

## 2. BPE a mano
Byte-pair encoding (diapositiva 38): empezar con caracteres, contar los pares vecinos en el texto de entrenamiento, fusionar el más frecuente en un símbolo nuevo y repetir. Aquí, 60 fusiones sobre un pedazo del libro, aplicadas a su primera frase.

In [ ]:
def pares(palabras):
    """Cuenta los pares de símbolos vecinos, pesando cada palabra por su frecuencia."""
    c = Counter()
    for simbolos, f in palabras.items():
        for a, b in zip(simbolos, simbolos[1:]):
            c[a, b] += f
    return c

def fusionar(palabras, par):
    nuevo = {}
    for simbolos, f in palabras.items():
        s, i = [], 0
        while i < len(simbolos):
            if i < len(simbolos) - 1 and (simbolos[i], simbolos[i + 1]) == par:
                s.append(simbolos[i] + simbolos[i + 1]); i += 2
            else:
                s.append(simbolos[i]); i += 1
        nuevo[tuple(s)] = f
    return nuevo

def segmentar(frase, fusiones):
    out = []
    for w in frase.split():
        s = list(w)
        for par in fusiones:
            s = list(fusionar({tuple(s): 1}, par))[0]
        out.append(s)
    return out

In [ ]:
muestra = texto_train[:200_000]                                       # un pedazo basta para ver las fusiones
palabras = {tuple(w): f for w, f in Counter(muestra.split()).items()}
fusiones = []
for k in range(1, 61):
    par = pares(palabras).most_common(1)[0][0]                        # el par más frecuente…
    palabras = fusionar(palabras, par)                                # …se vuelve un símbolo nuevo
    fusiones.append(par)
    if k in (1, 2, 3, 5, 10, 30, 60):
        seg = segmentar(FRASE, fusiones)
        print(f'{k:2d} fusiones · {sum(map(len, seg)):2d} símbolos · ' + ' '.join('|'.join(s) for s in seg))

Las primeras fusiones juntan pares muy frecuentes del español (terminaciones, artículos, "de", "que"); la frase se acorta con cada una. Las palabras frecuentes terminan siendo **un solo símbolo**; las raras quedan en pedazos, pero **nunca** son desconocidas: en el peor caso se escriben letra por letra.

## 3. BPE con `tokenizers`
La misma idea, rápida y sobre bytes (*byte-level*): los 256 bytes son el alfabeto inicial, así que cualquier texto, con tildes o emojis, se puede codificar. Comparamos cuatro tamaños de vocabulario por el número de tokens que necesita el texto de **validación**.

In [ ]:
def entrenar_bpe(n_vocab):
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)   # parte en bytes: nunca hay [UNK]
    tok.decoder = decoders.ByteLevel()
    tr = trainers.BpeTrainer(vocab_size=n_vocab, min_frequency=2, show_progress=False,
                             initial_alphabet=pre_tokenizers.ByteLevel.alphabet())
    tok.train_from_iterator([texto_train], tr)
    return tok

vocabularios, tokenizadores = {}, {}
for n in (300, 1000, 4000, 16000):
    tok = entrenar_bpe(n)
    ids = tok.encode(texto_val).ids
    tokenizadores[n] = tok
    vocabularios[n] = {'tokens_val': len(ids), 'chars_por_token': len(texto_val) / len(ids),
                       'frase': [tok.decode([i]) for i in tok.encode(FRASE).ids]}
    print(f'|V| = {n:6,d} · validación: {len(ids):8,d} tokens · {len(texto_val) / len(ids):.2f} caracteres por token')
    print('   ', ' | '.join(vocabularios[n]['frase']))

Más vocabulario, secuencias más cortas: el costo de la atención crece como $T^2$ (notebook 6.1), así que acortar importa. Pero cada token nuevo es una fila más de $E$ y de $W_U$, y los tokens raros casi no aparecen para aprenderlos. Usamos $\lvert V\rvert = 4\,000$. Los espacios van **pegados** a la palabra siguiente: " de" y "de" son tokens distintos.

In [ ]:
json.dump({'frase': FRASE, 'vocabularios': vocabularios,
           'fusiones_mini': [list(p) for p in fusiones[:60]], 'caracteres_val': len(texto_val)},
          open('bpe_vocabularios.json', 'w'), ensure_ascii=False, indent=1)

## 4. El GPT
El bloque pre-norm del notebook 6.2, con una diferencia: **solo** autoatención **causal**, sin codificador ni atención cruzada (el decodificador de la diapositiva 32). `F.scaled_dot_product_attention(..., is_causal=True)` construye la máscara. $W_U$ es la misma matriz que $E$ (diapositiva 29).

Los datos son ventanas de $T = 128$ tokens tomadas al azar; el objetivo es la misma ventana **corrida un lugar** (diapositiva 43).

In [ ]:
N_VOCAB = 4000
tok = tokenizadores[N_VOCAB] if 'tokenizadores' in globals() else entrenar_bpe(N_VOCAB)
ids_train = torch.tensor(tok.encode(texto_train).ids, dtype=torch.long)
ids_val = torch.tensor(tok.encode(texto_val).ids, dtype=torch.long)
print(f'{len(ids_train):,} tokens de entrenamiento · {len(ids_val):,} de validación')

def lote(ids, B, T):
    """B ventanas de T + 1 tokens tomadas al azar: la entrada son los primeros T, el objetivo los últimos T."""
    i = torch.randint(len(ids) - T - 1, (B,))
    x = torch.stack([ids[j:j + T] for j in i])
    y = torch.stack([ids[j + 1:j + T + 1] for j in i])               # el mismo texto, corrido un lugar
    return x.to(dev), y.to(dev)

In [ ]:
class Bloque(nn.Module):
    """El bloque pre-norm del notebook 6.2, solo con autoatención causal."""
    def __init__(self, d, H, p=0.1):
        super().__init__()
        self.H = H
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d)                                # W_Q, W_K, W_V juntas
        self.W_O = nn.Linear(d, d)
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
        self.drop, self.p = nn.Dropout(p), p

    def atencion(self, X):
        B, T, d = X.shape
        Q, K, V = self.qkv(X).view(B, T, 3, self.H, d // self.H).permute(2, 0, 3, 1, 4)   # (B, H, T, d_k) cada una
        Y = F.scaled_dot_product_attention(Q, K, V, is_causal=True,                       # máscara causal
                                           dropout_p=self.p if self.training else 0.0)
        return self.W_O(Y.transpose(1, 2).reshape(B, T, d))

    def forward(self, X):
        X = X + self.drop(self.atencion(self.ln1(X)))                # X ← X + MHA(LN(X))
        return X + self.drop(self.mlp(self.ln2(X)))                  # X ← X + MLP(LN(X))

class GPT(nn.Module):
    def __init__(self, n_vocab=N_VOCAB, d=192, H=6, D=4, T_max=256):
        super().__init__()
        self.T_max = T_max
        self.E = nn.Embedding(n_vocab, d)                             # embedding de tokens
        self.P = nn.Embedding(T_max, d)                               # p_i aprendidos
        self.bloques = nn.ModuleList([Bloque(d, H) for _ in range(D)])
        self.ln = nn.LayerNorm(d)
        self.W_U = nn.Linear(d, n_vocab, bias=False)
        self.W_U.weight = self.E.weight                               # atadas: la misma matriz (diapositiva 29)
        self.apply(self._iniciar)

    @staticmethod
    def _iniciar(m):                                                  # la inicialización de GPT-2: N(0, 0.02²)
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, ids):                                           # ids: (B, T)
        X = self.E(ids) + self.P(torch.arange(ids.shape[1], device=ids.device))
        for b in self.bloques:
            X = b(X)
        return self.W_U(self.ln(X))                                   # logits (B, T, |V|)

    @torch.no_grad()
    def generar(self, prompt, n=60, tau=1.0, top_k=None, greedy=False, seed=0):
        g = torch.Generator(device='cpu').manual_seed(seed)
        ids = torch.tensor([tok.encode(prompt).ids], device=dev)
        for _ in range(n):
            z = self(ids[:, -self.T_max:])[0, -1]                     # logits de la ÚLTIMA posición
            if greedy:
                sig = z.argmax()
            else:
                z = z / tau                                           # temperatura
                if top_k:
                    corte = z.topk(top_k).values[-1]
                    z = z.masked_fill(z < corte, float('-inf'))       # solo los k más probables
                sig = torch.multinomial(z.softmax(-1).cpu(), 1, generator=g)[0].to(dev)
            ids = torch.cat([ids, sig.view(1, 1)], 1)                 # el token producido es la siguiente entrada
        return tok.decode(ids[0].tolist())

**Predice antes de correr:** sin entrenar, ¿qué pérdida debería dar? Si el modelo no sabe nada, reparte la probabilidad por igual entre los $\lvert V\rvert$ tokens: $-\log(1/\lvert V\rvert) = \ln\lvert V\rvert$.

In [ ]:
torch.manual_seed(0)
gpt = GPT().to(dev)
n_par = sum(p.numel() for p in gpt.parameters())
print(f'{n_par:,} parámetros  (bloques: {4 * 12 * 192**2:,} = D·12d² sin sesgos · embeddings {N_VOCAB * 192:,}, compartidos con W_U)')
x, y = lote(ids_val, 8, 128)
with torch.no_grad():
    L0 = F.cross_entropy(gpt(x).flatten(0, 1), y.flatten()).item()
print(f'pérdida antes de entrenar {L0:.2f} · ln|V| = {math.log(N_VOCAB):.2f}')
print('muestra sin entrenar:', repr(gpt.generar('En un lugar de la Mancha', n=25)))

La pérdida inicial queda cerca de $\ln\lvert V\rvert$ gracias a la inicialización de GPT-2 ($\mathcal N(0, 0.02^2)$): con pesos más grandes, los logits iniciales serían enormes y la primera pérdida, mucho mayor (pruébalo en *Tu turno*). La muestra sin entrenar es basura, y a veces aparece un `�`: los tokens son bytes, y el modelo puede producir una secuencia que no es UTF-8 válido.

## 5. Entrenar unos minutos
AdamW, warmup y coseno, clipping (diapositiva 34). Cada paso: 64 ventanas de 128 tokens, es decir $64 \times 128$ predicciones y pérdidas **en una sola pasada**. Guardamos una muestra a los 200 pasos y otra al final.

**Predice** (diapositiva 47): ortografía, palabras, sintaxis, sentido. ¿Qué aparece primero? ¿Qué no llega con un modelo de este tamaño?

In [ ]:
@torch.no_grad()
def perdida_val(modelo, n=20, B=32, T=128):
    modelo.eval()
    torch.manual_seed(123)                                            # las mismas ventanas cada vez
    L = np.mean([F.cross_entropy(modelo(x).flatten(0, 1), y.flatten()).item()
                 for x, y in (lote(ids_val, B, T) for _ in range(n))])
    modelo.train()
    return L

def entrenar(modelo, pasos, B=64, T=128, lr=1e-3, warmup=100, cada=100, muestras_en=(), hist=None, paso0=0,
             opt=None, total=None):
    """J = la entropía cruzada media sobre las B·T posiciones: T predicciones por ventana, en una pasada."""
    total = total or pasos
    opt = opt or torch.optim.AdamW(modelo.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.1)
    f = lambda s: (s + 1) / warmup if s < warmup else 0.5 * (1 + math.cos(math.pi * (s - warmup) / max(1, total - warmup)))
    hist = hist if hist is not None else {'paso': [], 'train': [], 'val': [], 'muestras': {}}
    modelo.train()
    t0, acum = time.time(), []
    for s in range(paso0, paso0 + pasos):
        for gr in opt.param_groups:
            gr['lr'] = lr * f(s)                                      # warmup y coseno (sesión 02)
        x, y = lote(ids_train, B, T)
        loss = F.cross_entropy(modelo(x).flatten(0, 1), y.flatten())
        opt.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
        opt.step()
        acum.append(loss.item())
        if (s + 1) % cada == 0:
            hist['paso'].append(s + 1); hist['train'].append(float(np.mean(acum))); acum = []
            hist['val'].append(float(perdida_val(modelo)))
            print(f'paso {s + 1:6d} · train {hist["train"][-1]:.3f} · val {hist["val"][-1]:.3f} '
                  f'· {time.time() - t0:.0f} s')
        if (s + 1) in muestras_en:
            modelo.eval()
            hist['muestras'][s + 1] = modelo.generar('En un lugar de la Mancha', n=40, tau=0.8, seed=1)
            modelo.train()
            print('   muestra:', repr(hist['muestras'][s + 1]))
    return hist, opt

In [ ]:
PASOS = 2000 if dev == 'cuda' else 200                                # en CPU, solo un vistazo
torch.manual_seed(0)
gpt = GPT().to(dev)
hist, opt = entrenar(gpt, PASOS, muestras_en=(200, PASOS))

In [ ]:
plt.figure(figsize=(6.5, 3.2))
plt.plot(hist['paso'], hist['train'], label='entrenamiento')
plt.plot(hist['paso'], hist['val'], label='validación (últimos capítulos)')
plt.axhline(math.log(N_VOCAB), ls='--', color='gray', lw=1, label='ln|V|: adivinar al azar')
plt.xlabel('paso'); plt.ylabel('entropía cruzada (nats / token)'); plt.legend(); plt.tight_layout(); plt.show()
if hist['val']:
    print(f'perplejidad en validación: exp({hist["val"][-1]:.2f}) = {math.exp(hist["val"][-1]):.0f}')

Compara las dos muestras. Lo esperable (contrástalo con las tuyas): a los 200 pasos el modelo ya produce letras y espacios con frecuencias razonables y algunas palabras cortas; al final, palabras reales, puntuación y concordancias locales ("el caballero", "la venta"). El sentido a lo largo de varias frases no llega: para eso hace falta mucho más modelo y mucho más texto (Parte 4).

Mira también las dos curvas: si la de validación se separa hacia arriba de la de entrenamiento, el modelo empieza a memorizar el libro (sesión 02).

## 6. El *checkpoint* largo
El mismo modelo, entrenado mucho más tiempo con `course/s06_entrenar_gpt.ipynb`. El archivo trae su tokenizador: al cargarlo, el notebook usa ese.

In [ ]:
CKPT_HANDLE = 'juanmartinezv4399/si7011-s06-gpt-quijote'               # modelo de Kaggle con el checkpoint largo
CKPT_VARIANTE = 'pyTorch/default'

def kaggle_token():
    if 'KAGGLE_API_TOKEN' not in os.environ:
        try:
            from google.colab import userdata
            os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN').strip()
        except Exception:
            pass
    return 'KAGGLE_API_TOKEN' in os.environ or (Path.home() / '.kaggle' / 'access_token').exists()

ruta = buscar('gpt_quijote.pt')
if ruta is None and kaggle_token():
    try:
        import kagglehub
        ruta = next(Path(kagglehub.model_download(f'{CKPT_HANDLE}/{CKPT_VARIANTE}')).glob('**/gpt_quijote.pt'), None)
    except Exception as e:
        print('kagglehub:', type(e).__name__, '· no se pudo bajar', CKPT_HANDLE)

if ruta is not None:
    ck = torch.load(ruta, map_location=dev, weights_only=False)
    tok = Tokenizer.from_str(ck['tokenizador'])                       # el tokenizador con el que se entrenó
    largo = GPT(**ck['config']).to(dev)
    largo.load_state_dict(ck['modelo']); largo.eval()
    print(f"checkpoint: {ck['pasos']:,} pasos · val {ck['historia']['val'][-1]:.3f} "
          f"(perplejidad {math.exp(ck['historia']['val'][-1]):.0f})")
else:
    print('No hay checkpoint: las secciones siguientes usan el modelo de la sección 5.')
    largo, ck = gpt, None

In [ ]:
if ck is not None:
    h = ck['historia']
    plt.figure(figsize=(6.5, 3.2))
    plt.plot(h['paso'], h['val'], label='checkpoint largo, validación')
    plt.plot(hist['paso'], hist['val'], label='sección 5, validación')
    plt.xscale('log'); plt.xlabel('paso'); plt.ylabel('entropía cruzada'); plt.legend(); plt.tight_layout(); plt.show()
print(repr(largo.generar('En un lugar de la Mancha', n=60, tau=0.8, seed=1)))

## 7. Cómo elegir el token siguiente
Los mismos logits, cuatro reglas (diapositiva 46): **greedy** toma siempre el más probable; la **temperatura** $\tau$ divide los logits antes del softmax; **top-$k$** muestrea solo entre los $k$ más probables.

In [ ]:
PROMPT = 'Dijo Sancho a su amo'
print('greedy:      ', repr(largo.generar(PROMPT, n=50, greedy=True)))
for tau in (0.5, 1.0, 1.5):
    print(f'τ = {tau}:     ', repr(largo.generar(PROMPT, n=50, tau=tau, seed=2)))
print('top-k = 20:  ', repr(largo.generar(PROMPT, n=50, tau=1.0, top_k=20, seed=2)))

Greedy suele **repetirse**: si una frase lleva a sí misma, la sigue para siempre. $\tau$ pequeña se parece a greedy; $\tau$ grande inventa palabras. Top-$k$ corta la cola de tokens improbables, que son los que producen palabras rotas, sin quitar la variedad.

In [ ]:
json.dump({'pasos': hist['paso'], 'train': hist['train'], 'val': hist['val'],
           'muestras': {str(k): v for k, v in hist['muestras'].items()},
           'largo': None if ck is None else {'pasos': ck['pasos'], 'val': ck['historia']['val'][-1],
                                             'muestra': largo.generar('En un lugar de la Mancha', n=40, tau=0.8, seed=1)},
           'ln_V': math.log(N_VOCAB)},
          open('muestras_gpt.json', 'w'), ensure_ascii=False, indent=1)

## Tu turno · 10 minutos
1. En la clase `GPT`, quita `self.apply(self._iniciar)` y corre la sección 4. ¿Cuánto da la pérdida inicial ahora? ¿Por qué tanto, si el modelo es el mismo?
2. Entrena el tokenizador con $\lvert V\rvert = 300$ y el GPT con él (cambia `N_VOCAB`). ¿Cuántos tokens tiene una ventana de 128 en caracteres? ¿Cambia la calidad de las muestras con los mismos pasos?
3. Con el *checkpoint* largo, escribe un comienzo que **no** está en el libro ("El ingeniero de sistemas") y genera. ¿Qué conserva del estilo?
4. Busca en `texto_train` una frase que el modelo largo complete palabra por palabra. ¿Está memorizando? ¿Cómo lo distinguirías de haber aprendido el idioma?

**Para pensar.** El modelo aprendió solo con predecir el token siguiente. ¿Qué tendría que pasar para que, en vez de continuar un texto, **respondiera** una pregunta? (Parte 5 y ejercicio integrador.)

In [ ]:
# Tu turno

### Referencias
- Sennrich, Haddow y Birch (2016), *Neural Machine Translation of Rare Words with Subword Units*, ACL (BPE).
- Radford et al. (2019), *Language Models are Unsupervised Multitask Learners* (GPT-2: BPE sobre bytes, inicialización).
- Karpathy (2023), *nanoGPT* y *Let's build GPT: from scratch, in code, spelled out* (la estructura del modelo y del loop).
- Holtzman et al. (2020), *The Curious Case of Neural Text Degeneration*, ICLR (greedy, temperatura, top-$k$, top-$p$).
- Cervantes, *Don Quijote de la Mancha*, Project Gutenberg, libro 2000.